In [1]:
import json
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def extract_text_features(text):
    text_lower = text.lower()

    features = {
        "density": "normal",
        "region": "unspecified",
        "scarring": "absent",
        "pattern": "none",
        "severity": "unknown"
    }

    # Density
    if any(term in text_lower for term in [
        "ground-glass",
        "ground glass",
        "opacity",
        "opacities",
        "increased density",
        "increased interstitial markings",
        "increase in interstitial markings",
        "interstitial markings",
        "nodularity",
        "shaggy appearance",
        "shaggy outline",
        "indistinct outline",
        "less well delineated",
        "reticulation",
        "fibrosis",
        "fibrotic",
        "decreased gas exchange"
    ]):
        features["density"] = "increased"

    # Rules
    if any(term in text_lower for term in [
        "apico-basal",
        "apicobasal"
    ]):
        features["region"] = "apico_basal_lung"

    elif any(term in text_lower for term in [
        "adjacent lung",
        "adjacent to heart",
        "heart",
        "cardiac border"
    ]):
        features["region"] = "medial_lung"

    elif any(term in text_lower for term in [
        "entire lungs",
        "whole lungs",
        "both lungs",
        "bilateral",
        "diffuse"
    ]):
        features["region"] = "bilateral_lungs"

    elif any(term in text_lower for term in [
        "lower lobe",
        "lower lobes",
        "lower lung",
        "bibasal",
        "basal"
    ]):
        features["region"] = "lower_lung"

    elif any(term in text_lower for term in [
        "upper lobe",
        "upper lobes",
        "upper lung",
        "apical"
    ]):
        features["region"] = "upper_lung"

    elif any(term in text_lower for term in [
        "subpleural",
        "peripheral"
    ]):
        features["region"] = "peripheral_lung"

    elif "segmental" in text_lower:
        features["region"] = "segmental_lung"

    elif "lobar" in text_lower:
        features["region"] = "lobar_lung"

    # Scarring / fibrosis
    if any(term in text_lower for term in [
        "fibrosis",
        "fibrotic",
        "scarring",
        "scar",
        "reticulation",
        "reticular",
        "honeycombing",
        "uip",
        "usual interstitial pneumonia",
        "architectural distortion",
        "interlobular septal thickening",
        "interstitial markings",
        "restrictive pulmonary disease"
    ]):
        features["scarring"] = "present"

    # Pattern rules
    if any(term in text_lower for term in [
        "uip",
        "usual interstitial pneumonia"
    ]):
        features["pattern"] = "UIP_pattern"

    elif "honeycombing" in text_lower:
        features["pattern"] = "honeycombing"

    elif "traction bronchiectasis" in text_lower:
        features["pattern"] = "traction_bronchiectasis"

    elif "architectural distortion" in text_lower:
        features["pattern"] = "architectural_distortion"

    elif "interlobular septal thickening" in text_lower:
        features["pattern"] = "septal_thickening"

    elif any(term in text_lower for term in [
        "reticulation",
        "reticular"
    ]):
        features["pattern"] = "reticulation"

    elif "interstitial markings" in text_lower:
        features["pattern"] = "interstitial_markings"

    elif "nodularity" in text_lower:
        features["pattern"] = "nodularity"

    elif any(term in text_lower for term in [
        "fibrosis",
        "fibrotic"
    ]):
        features["pattern"] = "fibrosis"

    # Severity rules
    if any(term in text_lower for term in [
        "moderate to severe",
        "severe",
        "advanced",
        "extensive",
        "marked",
        "honeycombing",
        "long-standing",
        "long standing",
        "progressively",
        "progressive",
        "pulmonary arterial hypertension"
    ]):
        features["severity"] = "advanced"

    elif any(term in text_lower for term in [
        "moderate"
    ]):
        features["severity"] = "moderate"

    elif any(term in text_lower for term in [
        "mild",
        "early",
        "minimal",
        "subtle",
        "maintained lung volumes",
        "lung volumes are maintained"
    ]):
        features["severity"] = "early"

    return features


def map_features_to_3d(features):
    severity_map = {
        "early": {
            "opacity": 0.3,
            "deformation_intensity": 0.2
        },
        "moderate": {
            "opacity": 0.6,
            "deformation_intensity": 0.5
        },
        "advanced": {
            "opacity": 0.9,
            "deformation_intensity": 0.8
        },
        "unknown": {
            "opacity": 0.2,
            "deformation_intensity": 0.1
        }
    }

    params = severity_map.get(features["severity"], severity_map["unknown"])

    output = {
        **features,
        "opacity": params["opacity"],
        "deformation_intensity": params["deformation_intensity"],
        "texture": "scarred" if features["scarring"] == "present" else "normal"
    }

    return output


def process_radiology_text(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )

    with torch.no_grad():
        _ = model(**inputs)

    features = extract_text_features(text)
    final_output = map_features_to_3d(features)

    return final_output

text = """
This case shows pulmonary fibrosis with the apico-basal gradient and areas of honeycombing, forming a UIP pattern.
"""

result = process_radiology_text(text)

print(json.dumps(result, indent=4))

{
    "density": "increased",
    "region": "apico_basal_lung",
    "scarring": "present",
    "pattern": "UIP_pattern",
    "severity": "advanced",
    "opacity": 0.9,
    "deformation_intensity": 0.8,
    "texture": "scarred"
}
